# W02 Multi-Provider Evaluation — Leaderboard, Per-Platform Scores, Failure Modes, Cost x Quality

**InGen AI Model Evaluation · Week 2**

This notebook is the actual Week 2 deliverable referenced by the memo. It goes beyond
`data/leaderboard_summary.csv` in three ways that were missing from that file:

1. **Per-platform sub-scores** — the provider-wide leaderboard can hide a platform where a
   provider quietly underperforms.
2. **Full failure-mode distribution** — `leaderboard_summary.csv` only reports the single
   most common failure mode (`top_failure_mode`); this notebook reports every category.
3. **Cost x quality table + scatter** — combines `estimated_cost_usd`,
   `severity_weighted_score_norm`, and `mean_latency_ms` into one "best value" view.

All formulas for `severity_weighted_score`, `severity_weighted_score_norm`, and
`krippendorff_alpha` are imported unchanged from `week02_evaluation/leaderboard_analysis.py`
— the module that already produced `leaderboard_summary.csv` and was independently verified
against the judged data. This notebook does not re-derive or approximate them.

## 1. Data-Quality Guard

**This must run first and hard-fail (raise `AssertionError`) on any violation — no printed
warning, no bypass flag.** A prior notebook in this repo had a row-count assertion that was
silently skipped whenever a `dry_run` flag was set, which let a 6-row placeholder dataset pass
as if it were the real 40-row result. The checks below have no such escape hatch:

- (a) `judged_trackA_full_40*.json` and `judged_trackB_full_40*.json` each load to exactly
  80 rows (20 scenarios × 4 providers).
- (b) Every row has a non-null `judge_mean_accuracy` (i.e. `judge_coverage_pct == 100.0` for
  every provider). If not, the assertion prints the exact `(scenario_id, provider)` pairs
  that are missing and stops — the leaderboard cells below never run.
- (c) No row or file-level metadata carries a `dry_run` / `smoke_test` flag, and every row's
  `evaluation_set` is exactly `"full_40"`.

In [1]:
import sys
import glob
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

REPO_ROOT = Path().resolve().parents[0]
sys.path.insert(0, str(REPO_ROOT))

from week02_evaluation.multiprovider_eval.data_quality import run_guard
from week02_evaluation.leaderboard_analysis import (
    load_judged_dataframe, build_leaderboard, compute_krippendorff_alpha,
    ACCURACY_SEED_COLS, COST_PER_1K_TOKENS,
)
from week02_evaluation.multiprovider_eval.platform_scores import (
    build_platform_subscores, pivot_metric,
)
from week02_evaluation.multiprovider_eval.failure_modes import (
    failure_mode_distribution, ALL_CATEGORIES,
)
from week02_evaluation.multiprovider_eval.cost_quality import build_cost_quality_table

DATA_DIR = REPO_ROOT / "data"
trackA_paths = sorted(DATA_DIR.glob("judged_trackA_full_40*.json"))
trackB_paths = sorted(DATA_DIR.glob("judged_trackB_full_40*.json"))
assert trackA_paths, "No judged_trackA_full_40*.json found in data/"
assert trackB_paths, "No judged_trackB_full_40*.json found in data/"

judged_paths = [trackA_paths[-1], trackB_paths[-1]]

# Hard guard — raises AssertionError and stops the notebook on any violation.
loaded = run_guard(judged_paths)

for name, rows in loaded.items():
    print(f"[PASS] {name}: {len(rows)} rows, evaluation_set='full_40', "
          f"judge_coverage_pct=100.0 for every provider, no dry_run/smoke_test flags.")

[PASS] judged_trackA_full_40_20260724T050418Z.json: 80 rows, evaluation_set='full_40', judge_coverage_pct=100.0 for every provider, no dry_run/smoke_test flags.
[PASS] judged_trackB_full_40_20260724T050518Z.json: 80 rows, evaluation_set='full_40', judge_coverage_pct=100.0 for every provider, no dry_run/smoke_test flags.


## 2. Overall Provider Leaderboard

Reproduces `data/leaderboard_summary.csv` exactly — same `build_leaderboard()` function, same
`severity_weighted_score` / `severity_weighted_score_norm` / `krippendorff_alpha` formulas.
These were **independently verified** against the judged data before this notebook was built;
this cell re-derives them from the same source files rather than reading the CSV, so it also
serves as a live check that the CSV hasn't drifted from the underlying data.

In [2]:
df = load_judged_dataframe(judged_paths)
print(f"Combined rows (trackA + trackB): {len(df)}")

leaderboard = build_leaderboard(df)

alpha_accuracy = compute_krippendorff_alpha(df, ACCURACY_SEED_COLS, "ordinal")
print(f"Global Krippendorff alpha (task_accuracy, all providers pooled): {alpha_accuracy:.4f}")
print()

display_cols = [
    "provider", "n_scenarios", "n_scored", "judge_coverage_pct",
    "severity_weighted_score", "severity_weighted_score_norm",
    "mean_task_accuracy", "krippendorff_alpha", "estimated_cost_usd",
    "mean_latency_ms", "top_failure_mode",
]
leaderboard[display_cols]

15:43:29  INFO      Loading judged_trackA_full_40_20260724T050418Z.json
15:43:29  INFO      Loading judged_trackB_full_40_20260724T050518Z.json
15:43:29  INFO      Loaded 160 rows from 2 file(s)


Combined rows (trackA + trackB): 160
Global Krippendorff alpha (task_accuracy, all providers pooled): 0.7246



,provider,n_scenarios,n_scored,judge_coverage_pct,severity_weighted_score,severity_weighted_score_norm,mean_task_accuracy,krippendorff_alpha,estimated_cost_usd,mean_latency_ms,top_failure_mode
rank,,,,,,,,,,,
1,anthropic,40,40,100.0,364.67,0.9856,4.933,0.4958,0.0864,10705.2,correct
2,deepseek,40,40,100.0,340.00,0.9189,4.783,0.7598,0.0038,3185.2,correct
3,groq,40,40,100.0,330.67,0.8937,4.542,0.5671,0.0011,490.6,correct
4,openai,40,40,100.0,290.00,0.7838,4.467,0.8525,0.0602,2365.5,correct


In [3]:
fig = px.bar(
    leaderboard.reset_index(), x="provider", y="severity_weighted_score_norm",
    color="provider", text="severity_weighted_score_norm",
    title="Overall Severity-Weighted Score (normalized) by Provider",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(yaxis_range=[0, 1.05], showlegend=False, template="plotly_white")
fig.show()

## 3. Per-Platform Sub-Scores

For each of the 5 platforms (Fari, Senpai, Sentinel Prime AI, Aido Rover, Aido Humanoid) ×
4 providers: `n_scenarios_scored`, `mean_task_accuracy`, `severity_weighted_score`, and
`severity_weighted_score_norm`. **The normalizer here is platform-local**
(`sum(severity_class for that platform's scored rows) * 5`) — it is *not* the provider-wide
normalizer from section 2, so these numbers are not directly comparable to the overall
leaderboard, by design: a provider can look strong overall while quietly underperforming on
one platform, and that's exactly what this section is meant to surface.

In [4]:
platform_subscores = build_platform_subscores(df)
platform_subscores

,platform,provider,n_scenarios_scored,mean_task_accuracy,severity_weighted_score,severity_weighted_score_norm
0,Aido Humanoid,anthropic,8,4.667,74.666,0.9333
1,Aido Humanoid,deepseek,8,4.250,62.000,0.7750
2,Aido Humanoid,groq,8,4.500,68.666,0.8583
3,Aido Humanoid,openai,8,3.875,52.000,0.6500
4,Aido Rover,anthropic,8,5.000,65.000,1.0000
5,Aido Rover,deepseek,8,4.833,59.668,0.9180
6,Aido Rover,groq,8,4.500,59.001,0.9077
7,Aido Rover,openai,8,4.333,47.668,0.7334
8,Fari,anthropic,8,5.000,85.000,1.0000
9,Fari,deepseek,8,5.000,85.000,1.0000


In [5]:
pivot_norm = pivot_metric(platform_subscores, "severity_weighted_score_norm")
print("Pivot table — platform (rows) x provider (columns), severity_weighted_score_norm:")
pivot_norm

Pivot table — platform (rows) x provider (columns), severity_weighted_score_norm:


provider,anthropic,deepseek,groq,openai
platform,,,,
Aido Humanoid,0.9333,0.7750,0.8583,0.6500
Aido Rover,1.0000,0.9180,0.9077,0.7334
Fari,1.0000,1.0000,0.8824,1.0000
Senpai,1.0000,1.0000,0.9408,1.0000
Sentinel Prime AI,1.0000,0.9298,0.9018,0.6351


In [6]:
fig = px.imshow(
    pivot_norm,
    text_auto=".2f",
    color_continuous_scale="RdYlGn",
    zmin=0, zmax=1,
    labels=dict(x="provider", y="platform", color="severity_weighted_score_norm"),
    title="Per-Platform Severity-Weighted Score (normalized) — Heatmap",
    aspect="auto",
)
fig.update_layout(template="plotly_white")
fig.show()

## 4. Full Failure-Mode Distribution

`leaderboard_summary.csv` only reports `top_failure_mode` — the single most common category.
Below is the complete count/percentage breakdown across all categories judged
(`correct`, `hallucination`, `refusal`, `incomplete_response`, `safety_boundary_violation`,
`off_topic`, `generic_ungrounded`, `provider_error`), first overall per provider, then broken
down by platform × provider — a provider whose failures concentrate on one specific platform
is a more actionable finding than an aggregate rate.

In [7]:
overall_failure_dist = failure_mode_distribution(df, ["resp_provider"])
overall_pivot = overall_failure_dist.pivot(
    index="judge_majority_failure", columns="resp_provider", values="pct"
).reindex(ALL_CATEGORIES)
print("Failure-mode distribution by provider (% of that provider's scenarios):")
overall_pivot

Failure-mode distribution by provider (% of that provider's scenarios):


resp_provider,anthropic,deepseek,groq,openai
judge_majority_failure,,,,
correct,90.0,85.0,75.0,82.5
hallucination,0.0,0.0,0.0,0.0
refusal,5.0,10.0,2.5,12.5
incomplete_response,0.0,2.5,17.5,5.0
safety_boundary_violation,5.0,2.5,5.0,0.0
off_topic,0.0,0.0,0.0,0.0
generic_ungrounded,0.0,0.0,0.0,0.0
provider_error,0.0,0.0,0.0,0.0


In [8]:
platform_failure_dist = failure_mode_distribution(df, ["platform", "resp_provider"])

fig = px.bar(
    platform_failure_dist, x="resp_provider", y="count", color="judge_majority_failure",
    facet_col="platform", facet_col_wrap=3,
    category_orders={"judge_majority_failure": ALL_CATEGORIES},
    title="Failure-Mode Distribution by Provider, Faceted by Platform",
    labels={"resp_provider": "provider", "count": "# scenarios"},
)
fig.update_layout(template="plotly_white", height=700)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

## 5. Cost x Quality

`estimated_cost_usd` uses the same blended per-1K-token pricing constants already used to
build `leaderboard_summary.csv` (`COST_PER_1K_TOKENS` in `leaderboard_analysis.py`):

| provider  | USD / 1K tokens (blended input+output) | assumption |
|---|---|---|
| openai    | 0.005   | gpt-4o blended |
| anthropic | 0.004   | claude-sonnet-4-6 blended |
| deepseek  | 0.0003  | deepseek-chat blended |
| groq      | 0.00008 | llama-3.1-8b-instant blended |

`cost_per_quality_point = estimated_cost_usd / severity_weighted_score_norm` — lower is
better value. Ranked by `severity_weighted_score_norm` (quality), not by cost.

In [9]:
print("Pricing assumptions in effect (USD per 1K tokens, blended):")
for provider, price in COST_PER_1K_TOKENS.items():
    print(f"  {provider:10s} {price}")
print()

cost_quality = build_cost_quality_table(leaderboard)
cost_quality

Pricing assumptions in effect (USD per 1K tokens, blended):
  openai     0.005
  anthropic  0.004
  deepseek   0.0003
  groq       8e-05



,provider,severity_weighted_score_norm,estimated_cost_usd,mean_latency_ms,cost_per_quality_point
0,anthropic,0.9856,0.0864,10705.2,0.087662
1,deepseek,0.9189,0.0038,3185.2,0.004135
2,groq,0.8937,0.0011,490.6,0.001231
3,openai,0.7838,0.0602,2365.5,0.076805


In [10]:
fig = px.scatter(
    cost_quality, x="estimated_cost_usd", y="severity_weighted_score_norm",
    size="mean_latency_ms", color="provider", text="provider",
    size_max=40,
    title="Cost vs. Quality (point size = mean latency in ms)",
    labels={
        "estimated_cost_usd": "Estimated cost (USD, full 40-scenario run)",
        "severity_weighted_score_norm": "Severity-weighted score (normalized)",
    },
)
fig.update_traces(textposition="top center")
fig.update_layout(template="plotly_white", yaxis_range=[0, 1.05])
fig.show()

## Bottom Line

**Groq delivers ~91% of Anthropic's severity-weighted quality at ~1/78th the cost and under
5% of the latency — the clear best-value provider by `cost_per_quality_point` — but its
per-platform breakdown shows real weak spots on Aido Humanoid (0.86) and Fari (0.88), driven
by a 17.5% incomplete-response rate that the single aggregate leaderboard score hides; OpenAI,
despite being the second-most expensive provider, has the lowest overall quality of the four
(0.784) and its two worst platforms — Aido Humanoid (0.65) and Sentinel Prime AI (0.64) — are
exactly the safety-critical platforms where that matters most.**